This code is to take in embeddings extracted from a set of pre-trained models, perform dimension reduction, unsupervised clustering, and then test how the clustering across camera trap sites relates to species richness.

In [38]:
# packages
import os
import numpy as np
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import umap
from umap import plot
import json
import hdbscan
import sklearn.cluster as cluster
from collections import defaultdict
from sklearn import metrics, datasets
from sklearn.metrics import pairwise_distances, adjusted_rand_score, adjusted_mutual_info_score
import colorcet as cc
from pathlib import Path
import re


First we need to lead in the feature vectors from the output folder

In [39]:
os.chdir('/Users/peggybevan/Documents/Github/CV4E_unsupProj')

base_dir = Path('/Users/peggybevan/Documents/Github/CV4E_unsupProj')
output_dir = base_dir / 'output'
results_dir = base_dir / 'results'
results_dir.mkdir(parents=True, exist_ok=True)


In [40]:
# load metadata
#add information about camera trap location and species
meta = pd.read_csv('data/nepal_cropsmeta_PB.csv')
#meta has humans and vehicles in - remove
anthro = ['human', 'vehicle']
domes = ['buffalo', 'cow', 'dog', 'domestic_cat', 'domestic_chicken', 'domestic elephant', 'goat', 'sheep']
meta = meta[-meta['species'].isin(anthro)]
#only training
meta = meta[meta['SetID']=='train']
meta_wild = meta[-meta['species'].isin(domes)]

#create numpy array for important variables
ct_site = np.array(meta.ct_site)
species = np.array(meta.species)
mgmt = np.array(meta.conservancy_name) 
time_hour = np.array(meta.time_hour)

# and for wild only
ct_site_wild = np.array(meta_wild.ct_site)
species_wild = np.array(meta_wild.species)
mgmt_wild = np.array(meta_wild.conservancy_name)
time_hour_wild = np.array(meta_wild.time_hour)



In [41]:
# Dimensions to try
dims_list = [128, 32, 8, 2]

# HDBSCAN hyperparameters
# I'm using a leaf cluster selection method as this allows for smaller clusters to be identified 
# whilst still allowing variable cluster size.
# we will try some settings for min cluster size based on % of total samples
n_samples = meta_wild.shape[0]
n_samples_wild = meta_wild.shape[0]


hdbscan_cfgs = {
    # changing the minimum cluster size (interpreted as 0.5%, 1%, 2% of the total sample size)
    "leaf_0p5pct": lambda n: dict(
        min_cluster_size=int(0.005 * n),
        min_samples=(int(0.005 * n_samples) // 2),
        cluster_selection_method='leaf'
    ),
    "leaf_1pct": lambda n: dict(
        min_cluster_size=int(0.01 * n),
        min_samples=(int(0.01 * n_samples) // 2),
        cluster_selection_method='leaf'
    ),
    "leaf_2pct": lambda n: dict(
       min_cluster_size=int(0.02 * n),
        min_samples=(int(0.02 * n_samples) // 2),
        cluster_selection_method='leaf'
    )
}

hdbscan_cfgs_wild = {
    # changing the minimum cluster size (interpreted as 0.5%, 1%, 2% of the total sample size)
    "leaf_0p5pct": lambda n: dict(
        min_cluster_size=int(0.005 * n),
        min_samples=(int(0.005 * n_samples_wild) // 2),
        cluster_selection_method='leaf'
    ),
    "leaf_1pct": lambda n: dict(
        min_cluster_size=int(0.01 * n),
        min_samples=(int(0.01 * n_samples_wild) // 2),
        cluster_selection_method='leaf'
    ),
    "leaf_2pct": lambda n: dict(
       min_cluster_size=int(0.02 * n),
        min_samples=(int(0.02 * n_samples_wild) // 2),
        cluster_selection_method='leaf'
    )
}


List all the models in the output folder

In [42]:
# ---- Discover models in output/ ----
feature_files = sorted(output_dir.glob('*_fvect_norm.npy'))
if not feature_files:
    raise FileNotFoundError(f'No files matching "*_fvect_norm.npy" found in {output_dir}')

def infer_model_name(path: Path) -> str:
    # e.g., "PegNet50_fvect_norm.npy" -> "PegNet50"
    #  or "PegNet50_wild_fvect_norm.npy" -> "PegNet50_wild"
    return re.sub(r'_fvect_norm\.npy$', '', path.name)


In [44]:
# ---- Containers for results ----
site_rows = []       # one row per (model, n_components, ct_site)
regression_rows = [] # one row per (model, n_components)


In [ ]:
# run loop over all models and UMAP dimensions
# takes 30 mins to run for 4 models x 4 dims
for fpath in feature_files:
    model = infer_model_name(fpath)
    print(f'Processing model: {model}')

    # Load feature matrix
    features = np.load(fpath)
    n_samples = features.shape[0]
    
    # if model contains "wild", use the wild metadata and HDBSCAN configs
    if 'wild' in model:
        meta = meta_wild
        hdbscan_cfgs = hdbscan_cfgs_wild
        ct_site = ct_site_wild
        species = species_wild
    else:
        meta = meta
        hdbscan_cfgs = hdbscan_cfgs
        ct_site = ct_site
        species = species


    # Sanity check: feature rows must align with meta length
    if len(ct_site) != n_samples or len(species) != n_samples:
        raise ValueError(
            f'Length mismatch for model "{model}": '
            f'features={n_samples}, ct_site={len(ct_site)}, species={len(species)}.\n'
            'Ensure your meta filtering matches the feature set order, or provide an index mapping.'
        )

    for n_comp in dims_list:
        # Build and fit UMAP
        reducer = umap.UMAP(init='random', random_state=42, n_jobs = 1, n_components=n_comp)
        embedding = reducer.fit_transform(features)

        # Cluster with HDBSCAN
        for cfg_name, cfg_fn in hdbscan_cfgs.items():
            # Build HDBSCAN kwargs for this dataset size
            kwargs = cfg_fn(n_samples)

            # Fit HDBSCAN
            clusterer = hdbscan.HDBSCAN(**kwargs).fit(embedding)
            labels = clusterer.labels_

            # Diagnostics
            n_clusters = int(labels.max() + 1) if labels.max() >= 0 else 0  # exclude noise (-1)
            noise_ratio = float((labels == -1).mean())

            # Per-site counts (remove noise label -1 when counting clusters)
            distinct_label_counts = []
            distinct_species_counts = []

            for site in np.unique(ct_site):
                indices = np.where(ct_site == site)[0]
                site_labels = labels[indices]
                site_species = species[indices]

                site_labels = site_labels[site_labels != -1]
                distinct_label_count = int(len(np.unique(site_labels)))
                distinct_species_count = int(len(np.unique(site_species)))

                site_rows.append({
                    'model': model,
                    'n_components': n_comp,
                    'hdbscan_cfg': cfg_name,
                    'min_cluster_size': int(kwargs['min_cluster_size']),
                    'min_samples': int(kwargs['min_samples']),
                    'ct_site': site,
                    'distinct_label_count': distinct_label_count,
                    'distinct_species_count': distinct_species_count,
                    'n_samples_site': int(len(indices)),
                    'n_clusters_total': n_clusters,
                    'noise_ratio_total': noise_ratio
                })

                distinct_label_counts.append(distinct_label_count)
                distinct_species_counts.append(distinct_species_count)

            # Regression summaries across sites
            x = np.array(distinct_label_counts, dtype=float)
            y = np.array(distinct_species_counts, dtype=float)

            # Guard against zero-variance arrays (otherwise polyfit/corr can be nan)
            if np.all(x == x[0]) or np.all(y == y[0]):
                slope, intercept = (np.nan, np.nan)
                corr = np.nan
                r2 = np.nan
            else:
                slope, intercept = np.polyfit(x, y, 1)
                corr = np.corrcoef(x, y)[0, 1]
                # Simple r^2 from correlation
                r2 = corr**2

            regression_rows.append({
                'model': model,
                'n_components': n_comp,
                'hdbscan_cfg': cfg_name,
                'min_cluster_size': int(kwargs['min_cluster_size']),
                'min_samples': int(kwargs['min_samples']),
                'slope': float(slope) if slope == slope else np.nan,   # keep as float or NaN
                'intercept': float(intercept) if intercept == intercept else np.nan,
                'correlation_r': float(corr) if corr == corr else np.nan,
                'r_squared': float(r2) if r2 == r2 else np.nan,
                'n_sites': int(len(np.unique(ct_site_wild))),
                'n_samples_total': int(n_samples),
                'n_clusters_global': n_clusters,
                'noise_ratio_global': noise_ratio
                })


Processing model: PegNet50


In [31]:
#  ---- Build DataFrames ----
site_counts_df = pd.DataFrame(site_rows)
regression_df = pd.DataFrame(regression_rows)


In [ ]:
regression_df

In [ ]:

site_csv = results_dir / 'clustercountsbysite.csv'
regr_csv = results_dir / 'regression_summary.csv'

site_counts_df.to_csv(site_csv, index=False)
regression_df.to_csv(regr_csv, index=False)


In [34]:
# plot

sns.set(style="whitegrid", context="talk")

# Ensure categorical hue/style mapping (so seaborn uses discrete colors/dashes)
for df in (site_counts_df, regression_df):
    df["n_components"] = df["n_components"].astype("category")
    df["hdbscan_cfg"] = df["hdbscan_cfg"].astype("category")


In [36]:

limits_df = (
    site_counts_df.groupby(["model", "n_components", "hdbscan_cfg"])
    .agg(xmin=("distinct_label_count", "min"), xmax=("distinct_label_count", "max"))
    .reset_index()
)

lines_base = regression_df.merge(
    limits_df, on=["model", "n_components", "hdbscan_cfg"], how="left"
)

# Expand each line to 50 points between xmin and xmax
line_rows = []
for _, row in lines_base.iterrows():
    if any(pd.isna(row[k]) for k in ["slope", "intercept", "xmin", "xmax"]):
        continue
    # Skip degenerate ranges
    if row["xmax"] <= row["xmin"]:
        continue
    xs = np.linspace(row["xmin"], row["xmax"], 50)
    ys = row["slope"] * xs + row["intercept"]
    line_rows.append(pd.DataFrame({
        "model": row["model"],
        "n_components": row["n_components"],
        "hdbscan_cfg": row["hdbscan_cfg"],
        "x": xs,
        "y": ys
    }))
line_points_df = pd.concat(line_rows, ignore_index=True) if line_rows else pd.DataFrame()

# Optional: consistent dash patterns for style levels
style_order = ["leaf_0p5pct", "leaf_1pct", "leaf_2pct"]
dashes_map = {
    "leaf_0p5pct": (2, 2),   # short dash
    "leaf_1pct": (6, 2),     # medium dash
    "leaf_2pct": (10, 3),    # long dash
}


/var/folders/d3/rn8l2fts0xxbcbrzl7m0hvv00000gn/T/ipykernel_64342/2199339047.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  site_counts_df.groupby(["model", "n_components", "hdbscan_cfg"])


In [37]:

# ----- Scatter facets -----
g = sns.relplot(
    data=site_counts_df,
    x="distinct_label_count", y="distinct_species_count",
    col="model", col_wrap=3,
    hue="n_components", palette="tab10",
    style="hdbscan_cfg", style_order=style_order,
    kind="scatter", height=4, aspect=1.15,
    facet_kws={"sharex": False, "sharey": False},
    alpha=0.4
)

# After relplot
# Remove scatter points from the relplot facetsn lines on each facet -----
for ax in g.axes.flatten(): g.col_names):
    for coll in list(ax.collections):odel"] == model]
        coll.remove()

# ----- Overlay regression lines on each facet -----            data=sub,
for i, (ax, model) in enumerate(zip(g.axes.flatten(), g.col_names)):            x="x", y="y",
    sub = line_points_df[line_points_df["model"] == model]
    if not sub.empty:rder,
        sns.lineplot(
            data=sub,=2, ax=ax,
            x="x", y="y",lse  # keep legend from the scatter only
            hue="n_components", palette="tab10",
            style="hdbscan_cfg", style_order=style_order,}")
            dashes=dashes_map,")
            linewidth=2, ax=ax,
            legend=(i == 0)  # build legend once
        )ut using public seaborn API
    ax.set_title(f"{model}")
    ax.set_xlabel("Distinct Clusters per CT site")ove_legend(g, "lower right", title="n_components / HDBSCAN")
    ax.set_ylabel("Species Count per CT site")eError):

# Move legend to figure-level (no private _legend access)
handles, labels = g.axes.flatten()[0].get_legend_handles_labels()plt.show()
if handles:
    dedup = {}
    for h, l in zip(handles, labels):        if l and l not in dedup:
            dedup[l] = h
    g.figure.legend(
        dedup.values(), dedup.keys(),
        title="n_components / HDBSCAN",
        loc="lower right"
    )
    if g.axes.flatten()[0].legend_ is not None:
        g.axes.flatten()[0].legend_.remove()

SyntaxError: unmatched ')' (1180502740.py, line 15)